In [ ]:
!pip -q install "transformers==5.17.0" "accelerate==1.14.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 1.7 MB/s eta 0:00:00


In [ ]:
## Cell 0 — 부트스트랩
!nvidia-smi --query-gpu=name,memory.total --format=csv

from google.colab import drive, userdata
drive.mount('/content/drive')
import os, json, hashlib, numpy as np, pandas as pd, torch, transformers
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

!wget -q -O p8s_prompt.py https://raw.githubusercontent.com/Ontology0/Graduation-Project/refs/heads/dev/prompt/p8s_prompt.py
PROMPT_MD5 = hashlib.md5(open("p8s_prompt.py","rb").read()).hexdigest()
from p8s_prompt import build_messages, PROMPT_VERSION, MODEL_ID
assert PROMPT_MD5 == "20959a4c34203ea0866d030baf23595d", "v2 파일이 아님"

P9  = "/content/drive/MyDrive/Grad_share/data_share/P9/"
df = pd.read_csv("/content/drive/MyDrive/Grad_share/data_share/데이터셋 모음/p8s_pairs.csv")
os.makedirs(P9 + "hidden_v2", exist_ok=True)

from transformers import AutoTokenizer, AutoModelForCausalLM
REV = "0e9e39f249a16976918f6564b8830bc894c89659"
tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=REV, token=os.environ["HF_TOKEN"])
tok.padding_side = "left"
if tok.pad_token is None: tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REV, dtype=torch.bfloat16,
        device_map="auto", attn_implementation="sdpa", token=os.environ["HF_TOKEN"]).eval()

print(PROMPT_VERSION, df.shape, len(model.model.layers), transformers.__version__)

name, memory.total [MiB]
NVIDIA L4, 23034 MiB
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

p8s-v2 (1224, 16) 32 5.17.0


In [ ]:
## meta 복원 (C1-5에서 저장하던 것만)
NL, D = len(model.model.layers), model.config.hidden_size
ln_w = np.stack([model.model.layers[l].post_attention_layernorm.weight.detach().float().cpu().numpy() for l in range(NL)])
np.save(P9+"hidden_v2/p8s_v2_post_attn_ln_weight.npy", ln_w.astype(np.float32))
meta = dict(prompt_version=PROMPT_VERSION, prompt_md5=PROMPT_MD5, model_id=MODEL_ID,
    transformers_version=transformers.__version__, torch_version=torch.__version__,
    dtype_compute="bfloat16", dtype_stored="float16", n_layers=NL, hidden_size=D, batch_size=16,
    layer_index="0-based; index i == model.model.layers[i]",
    position="last prefill token (left padding, [:, -1, :])",
    row_order="p8s_behavior.csv의 행 순서 그대로 (0..1223)", streams=["h_out","attn_out","mlp_out"],
    reconstruct=dict(h_mid="h_out[l] - mlp_out[l]",
        h_in="h_out[l] - mlp_out[l] - attn_out[l]  (l>=1이면 h_out[l-1]과 일치)",
        mlp_in="RMSNorm(h_mid[l]) * post_attn_ln_weight[l], eps=rms_norm_eps"),
    rms_norm_eps=float(model.config.rms_norm_eps),
    smoke=dict(residual_relRMS=2.69e-3, h_mid_relRMS=1.78e-3,
        note="bf16 정밀 한계 3.9e-3 이내. fp16 저장 시 손실 없음(2.69e-3 동일)"))
json.dump(meta, open(P9+"hidden_v2/p8s_v2_meta.json","w"), ensure_ascii=False, indent=2)
print(sorted(os.listdir(P9+"hidden_v2")))

['p8s_v2_meta.json', 'p8s_v2_post_attn_ln_weight.npy']


In [ ]:
## Cell C1-1 — 캡처 훅
import torch, numpy as np, json

class Capturer:
    """decoder layer의 h_out / attn_out / mlp_out을 마지막 prefill 토큰에서 회수.

    복원 (l=0 포함):
        h_mid[l] = h_out[l] - mlp_out[l]
        h_in[l]  = h_mid[l] - attn_out[l]
    l>=1에서 h_in[l] == h_out[l-1] 이어야 한다 (정합성 검증식).
    """
    def __init__(self, model, extra_mid=False):
        self.m = model
        self.L = len(model.model.layers)
        self.extra_mid = extra_mid      # 스모크 전용: h_mid / mlp_in 직접 캡처
        self.buf, self.handles = {}, []

    def clear(self):
        self.buf.clear()

    def _mk(self, key, l):
        def hook(mod, args, out):
            t = out[0] if isinstance(out, (tuple, list)) else out
            if not torch.is_tensor(t) or t.ndim != 3:
                raise RuntimeError(f"{key}[{l}] 예상 밖 출력: type={type(out)}, "
                                   f"shape={getattr(t,'shape',None)}")
            self.buf[(key, l)] = t[:, -1, :].detach().float().cpu()
        return hook

    def _mk_ln(self, l):
        """post_attention_layernorm: 입력=h_mid, 출력=mlp_in. 둘 다 잡는다."""
        def hook(mod, args, out):
            self.buf[("h_mid", l)]  = args[0][:, -1, :].detach().float().cpu()
            self.buf[("mlp_in", l)] = out[:, -1, :].detach().float().cpu()
        return hook

    def __enter__(self):
        for l in range(self.L):
            lay = self.m.model.layers[l]
            self.handles += [
                lay.register_forward_hook(self._mk("h_out", l)),
                lay.self_attn.register_forward_hook(self._mk("attn_out", l)),
                lay.mlp.register_forward_hook(self._mk("mlp_out", l)),
            ]
            if self.extra_mid:
                self.handles.append(
                    lay.post_attention_layernorm.register_forward_hook(self._mk_ln(l)))
        return self

    def __exit__(self, *a):
        for h in self.handles: h.remove()
        self.handles = []

    def stack(self, keys=("h_out","attn_out","mlp_out")):
        return {k: torch.stack([self.buf[(k, l)] for l in range(self.L)], 1).numpy()
                for k in keys}

In [ ]:
## Cell C1-2 — 스모크 50건 (v2 프롬프트)
from p8s_prompt import build_messages, PROMPT_VERSION

assert tok.padding_side == "left", f"padding_side={tok.padding_side}"
assert tok.pad_token_id is not None

model.eval()
smoke = df.sample(50, random_state=42).copy()
smoke["_orig_row"] = smoke.index
smoke = smoke.reset_index(drop=True)

def capture(prompt_list, extra_mid=True, bs=16):
    outs = []
    KEYS = ("h_out","attn_out","mlp_out") + (("h_mid","mlp_in") if extra_mid else ())
    with torch.inference_mode(), Capturer(model, extra_mid=extra_mid) as cap:
        for s in range(0, len(prompt_list), bs):
            cap.clear()
            enc = tok(prompt_list[s:s+bs], return_tensors="pt",
                      padding=True, add_special_tokens=False).to(model.device)
            assert torch.all(enc["attention_mask"][:, -1] == 1), "마지막 위치에 padding 있음"
            _ = model.model(**enc, use_cache=False, return_dict=True)
            outs.append(cap.stack(KEYS))
    return {k: np.concatenate([o[k] for o in outs]) for k in KEYS}

p_v2 = [tok.apply_chat_template(build_messages(r.question, r.ctx_text),
        tokenize=False, add_generation_prompt=True) for r in smoke.itertuples()]
H = capture(p_v2)
print({k: v.shape for k, v in H.items()})

{'h_out': (50, 32, 4096), 'attn_out': (50, 32, 4096), 'mlp_out': (50, 32, 4096), 'h_mid': (50, 32, 4096), 'mlp_in': (50, 32, 4096)}


In [ ]:
## Cell C1-3 — 정합성 검증
def rel_rms(a, b):
    return np.sqrt(np.mean((a-b)**2)) / np.sqrt(np.mean(b**2))

# ① 복원 h_mid vs 직접 캡처 h_mid (전 레이어)
mid_rec = H["h_out"] - H["mlp_out"]
print(f"[h_mid] relRMS={rel_rms(mid_rec, H['h_mid']):.2e}  "
      f"max={np.abs(mid_rec-H['h_mid']).max():.4f}")

# ② 잔차 항등식: h_in[l] == h_out[l-1]
h_in = mid_rec - H["attn_out"]
res  = h_in[:,1:] - H["h_out"][:,:-1]
print(f"[residual] relRMS={rel_rms(h_in[:,1:], H['h_out'][:,:-1]):.2e}  "
      f"q99.9={np.quantile(np.abs(res),0.999):.4f}  max={np.abs(res).max():.4f}")

# ③ 레이어별 — 특정 레이어만 튀는지
per_l = [rel_rms(h_in[:,l], H["h_out"][:,l-1]) for l in range(1, H["h_out"].shape[1])]
print("layer-wise relRMS  min/med/max:",
      f"{min(per_l):.2e} / {np.median(per_l):.2e} / {max(per_l):.2e}",
      f"(worst=L{int(np.argmax(per_l))+1})")

# ④ fp16 저장 시뮬레이션 — cancellation 포함
H16 = {}
for k in ("h_out","attn_out","mlp_out"):
    with np.errstate(over="ignore", invalid="ignore"):
        v16 = H[k].astype(np.float16)
    H16[k] = v16.astype(np.float32)
    print(f"{k:9s} absmax={np.abs(H[k]).max():8.1f}  fp16_finite={np.isfinite(v16).all()}")

mid16 = H16["h_out"] - H16["mlp_out"]
print(f"[fp16 h_mid]   relRMS={rel_rms(mid16, H['h_mid']):.2e}")
res16 = (mid16 - H16["attn_out"])[:,1:] - H16["h_out"][:,:-1]
print(f"[fp16 residual] relRMS={rel_rms((mid16-H16['attn_out'])[:,1:], H16['h_out'][:,:-1]):.2e}")

[h_mid] relRMS=1.78e-03  max=0.0781
[residual] relRMS=2.69e-03  q99.9=0.0059  max=0.1875
layer-wise relRMS  min/med/max: 2.43e-03 / 2.75e-03 / 3.83e-03 (worst=L2)
h_out     absmax=    33.0  fp16_finite=True
attn_out  absmax=    10.3  fp16_finite=True
mlp_out   absmax=    42.2  fp16_finite=True
[fp16 h_mid]   relRMS=1.78e-03
[fp16 residual] relRMS=2.69e-03


In [ ]:
## Cell C1-4 — 훅 검증 (v1 프롬프트 ↔ 기존 hid_p8s.npy)
V1_SYS = ("You are a question answering system. "
          "Answer the question using ONLY the given context. "
          "Output the answer string itself and nothing else: no sentence, no explanation, "
          "no punctuation, no quotes. Keep it as short as possible (a name or a term). "
          "If the context does not state the answer, output the most likely short answer anyway.")

V1_FEWSHOT = [
    {"context": ("Reykjavik Reykjavik is the capital and largest city of Iceland. "
                 "Located in the south-western part of the island, it is the seat of the "
                 "national government and the country's main cultural centre."),
     "question": "What is the capital of Iceland?", "answer": "Reykjavik"},
    {"context": ("The Old Man and the Sea The Old Man and the Sea is a short novel written by "
                 "Ernest Hemingway in 1951 in Cuba and published in 1952. It was the last major "
                 "work of fiction produced by Hemingway during his lifetime."),
     "question": "Who is the author of The Old Man and the Sea?", "answer": "Ernest Hemingway"},
    {"context": ("Ada Lovelace Augusta Ada King, Countess of Lovelace, was an English mathematician "
                 "chiefly known for her work on Charles Babbage's Analytical Engine. She is often "
                 "regarded as the first computer programmer."),
     "question": "What is Ada Lovelace's occupation?", "answer": "mathematician"},
]

def v1_msgs(q, ctx):
    m = [{"role":"system","content":V1_SYS}]
    for ex in V1_FEWSHOT:
        m += [{"role":"user","content":f"Context: {ex['context']}\nQuestion: {ex['question']}\nAnswer:"},
              {"role":"assistant","content":ex["answer"]}]
    m.append({"role":"user","content":f"Context: {ctx}\nQuestion: {q}\nAnswer:"})
    return m

p_v1 = [tok.apply_chat_template(v1_msgs(r.question, r.ctx_text),
        tokenize=False, add_generation_prompt=True) for r in smoke.itertuples()]
Hv1 = capture(p_v1, extra_mid=False)

old = np.load(P9+"hid_p8s.npy", mmap_mode="r")
raw = np.asarray(old[smoke._orig_row.to_numpy()])
print("old dtype/shape:", old.dtype, old.shape)
for name, cand in [("L-major", raw.reshape(50,32,4096)),
                   ("D-major", raw.reshape(50,4096,32).transpose(0,2,1))]:
    print(f"[{name}] relRMS={rel_rms(Hv1['h_out'], cand):.2e}")

print(f"[v2 vs v1] relRMS={rel_rms(H['h_out'], Hv1['h_out']):.2e}")

old dtype/shape: float32 (1224, 131072)
[L-major] relRMS=7.82e-01
[D-major] relRMS=1.24e+00
[v2 vs v1] relRMS=4.63e-01


In [ ]:
## Cell C1-4b — 레이아웃 확정 (프롬프트 무관)
for name, cand in [("L-major", raw.reshape(50,32,4096)),
                   ("D-major", raw.reshape(50,4096,32).transpose(0,2,1))]:
    nrm = np.linalg.norm(cand, axis=2).mean(0)
    print(f"[{name}] L1={nrm[0]:.1f} L8={nrm[7]:.1f} L16={nrm[15]:.1f} "
          f"L24={nrm[23]:.1f} L32={nrm[31]:.1f}  단조={bool(np.all(np.diff(nrm)>0))}")

nrm_new = np.linalg.norm(Hv1["h_out"], axis=2).mean(0)
print(f"[new]     L1={nrm_new[0]:.1f} L8={nrm_new[7]:.1f} L16={nrm_new[15]:.1f} "
      f"L24={nrm_new[23]:.1f} L32={nrm_new[31]:.1f}")

[L-major] L1=0.8 L8=4.4 L16=8.5 L24=19.8 L32=139.9  단조=True
[D-major] L1=32.6 L8=26.2 L16=28.9 L24=30.3 L32=28.2  단조=False
[new]     L1=0.8 L8=4.8 L16=8.9 L24=21.9 L32=63.1


In [ ]:
## Cell C1-5 — 전체 1,224건 추출
import time, shutil

LOCAL = "/content/hid_v2/"; os.makedirs(LOCAL, exist_ok=True)
KEYS  = ("h_out", "attn_out", "mlp_out")
NL    = len(model.model.layers)
D     = model.config.hidden_size
BS    = 16

prompts = [tok.apply_chat_template(build_messages(r.question, r.ctx_text),
           tokenize=False, add_generation_prompt=True) for r in df.itertuples()]

stores = {k: np.lib.format.open_memmap(LOCAL+f"p8s_v2_{k}.npy", mode="w+",
          dtype=np.float16, shape=(len(df), NL, D)) for k in KEYS}

t0 = time.time()
with torch.inference_mode(), Capturer(model, extra_mid=False) as cap:
    for s in range(0, len(prompts), BS):
        cap.clear()
        enc = tok(prompts[s:s+BS], return_tensors="pt",
                  padding=True, add_special_tokens=False).to(model.device)
        assert torch.all(enc["attention_mask"][:, -1] == 1)
        _ = model.model(**enc, use_cache=False, return_dict=True)
        b = cap.stack(KEYS)
        for k in KEYS:
            stores[k][s:s+enc["input_ids"].shape[0]] = b[k].astype(np.float16)
        if s % (BS*20) == 0:
            print(f"  {s}/{len(prompts)}  {time.time()-t0:.0f}s", flush=True)

for k in KEYS: stores[k].flush()
print(f"done {time.time()-t0:.0f}s")

# mlp_in 복원용: post_attention_layernorm 가중치 + eps
ln_w = np.stack([model.model.layers[l].post_attention_layernorm.weight
                 .detach().float().cpu().numpy() for l in range(NL)])
np.save(LOCAL+"p8s_v2_post_attn_ln_weight.npy", ln_w.astype(np.float32))

meta = dict(
    prompt_version=PROMPT_VERSION, prompt_md5=PROMPT_MD5,
    model_id=MODEL_ID, transformers_version=transformers.__version__,
    torch_version=torch.__version__, dtype_compute="bfloat16", dtype_stored="float16",
    n_layers=NL, hidden_size=D, batch_size=BS,
    layer_index="0-based; index i == model.model.layers[i]",
    position="last prefill token (left padding, [:, -1, :])",
    row_order="p8s_behavior.csv의 행 순서 그대로 (0..1223)",
    streams=list(KEYS),
    reconstruct=dict(
        h_mid="h_out[l] - mlp_out[l]",
        h_in="h_out[l] - mlp_out[l] - attn_out[l]  (l>=1이면 h_out[l-1]과 일치)",
        mlp_in="RMSNorm(h_mid[l]) * post_attn_ln_weight[l], eps=rms_norm_eps"),
    rms_norm_eps=float(model.config.rms_norm_eps),
    smoke=dict(residual_relRMS=2.69e-3, h_mid_relRMS=1.78e-3,
               note="bf16 정밀 한계 3.9e-3 이내. fp16 저장 시 손실 없음(2.69e-3 동일)"),
)
json.dump(meta, open(LOCAL+"p8s_v2_meta.json","w"), ensure_ascii=False, indent=2)

DST = P9 + "hidden_v2/"; os.makedirs(DST, exist_ok=True)
for f in os.listdir(LOCAL): shutil.copy(LOCAL+f, DST+f)
print(sorted(os.listdir(DST)))

  0/1224  2s
  320/1224  32s
  640/1224  61s
  960/1224  91s
done 114s
['p8s_v2_attn_out.npy', 'p8s_v2_h_out.npy', 'p8s_v2_meta.json', 'p8s_v2_mlp_out.npy', 'p8s_v2_post_attn_ln_weight.npy']


In [ ]:
## Cell C1-6 — 저장본 검증
Hf = {k: np.load(DST+f"p8s_v2_{k}.npy", mmap_mode="r") for k in KEYS}
i = np.random.RandomState(0).choice(len(df), 200, replace=False)
A = {k: np.asarray(Hf[k][np.sort(i)]).astype(np.float32) for k in KEYS}

mid = A["h_out"] - A["mlp_out"]
hin = mid - A["attn_out"]
print(f"residual relRMS = {rel_rms(hin[:,1:], A['h_out'][:,:-1]):.2e}   (스모크 2.69e-03)")
print("finite:", {k: bool(np.isfinite(v).all()) for k, v in A.items()})
print("absmax:", {k: round(float(np.abs(v).max()),1) for k, v in A.items()})
nrm = np.linalg.norm(A["h_out"], axis=2).mean(0)
print("norm L1/L8/L16/L24/L32:", [round(float(nrm[j]),1) for j in (0,7,15,23,31)])

residual relRMS = 2.72e-03   (스모크 2.69e-03)
finite: {'h_out': True, 'attn_out': True, 'mlp_out': True}
absmax: {'h_out': 34.2, 'attn_out': 11.1, 'mlp_out': 44.0}
norm L1/L8/L16/L24/L32: [0.8, 4.4, 8.7, 21.8, 58.9]


## K+, K-

In [ ]:
## 배치 앞에 한 번
import gc
gc.collect(); torch.cuda.empty_cache()
print(f"{torch.cuda.memory_allocated()/1e9:.1f}GB 사용 중")

16.7GB 사용 중


In [ ]:
import gc
for v in ["H","Hv1","A","Hf","stores","raw","old_s","b","enc","out","g","q","cap"]:
    if v in globals(): del globals()[v]
gc.collect(); torch.cuda.empty_cache()
print(f"{torch.cuda.memory_allocated()/1e9:.1f}GB")

16.7GB


In [ ]:
## Cell K-1 — closed-book 재판정 (greedy 라벨 + 샘플 5회 안정성)
import time
from p8s_prompt import build_messages_cb, truncate_answer, score_row, MAX_NEW_TOKENS

DECODE_KW = dict(skip_special_tokens=True, clean_up_tokenization_spaces=False)
EOS_IDS = {tok.eos_token_id, tok.convert_tokens_to_ids("<|eot_id|>")} - {None}

df["gold_list"] = df["gold"].apply(json.loads)
qs = df.drop_duplicates("question")[["question","gold_list","kside","knows"]].reset_index(drop=True)
cb = [tok.apply_chat_template(build_messages_cb(q), tokenize=False,
      add_generation_prompt=True) for q in qs.question]
print(len(qs))

@torch.no_grad()
def gen(idx, **kw):
    enc = tok([cb[i] for i in idx], return_tensors="pt",
              padding=True, add_special_tokens=False).to(model.device)
    out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS,
                         pad_token_id=tok.pad_token_id,
                         eos_token_id=list(EOS_IDS), use_cache=True, **kw)
    return out[:, enc["input_ids"].shape[1]:]

## Cell K-1b — 생성 + 채점 + 저장
BS_G, BS_S = 32, 8

torch.manual_seed(42)
greedy, samp, t0 = [], [], time.time()

for s in range(0, len(cb), BS_G):
    idx = list(range(s, min(s+BS_G, len(cb))))
    g = gen(idx, do_sample=False)
    greedy += [tok.decode(x, **DECODE_KW) for x in g]
    del g

    for t in range(0, len(idx), BS_S):
        sub = idx[t:t+BS_S]
        q = gen(sub, do_sample=True, temperature=0.7, top_p=0.9, num_return_sequences=5)
        samp += [[tok.decode(x, **DECODE_KW) for x in q[j*5:(j+1)*5]] for j in range(len(sub))]
        del q

    if s % 160 == 0:
        print(f"  {s}/{len(cb)} {time.time()-t0:.0f}s "
              f"{torch.cuda.memory_allocated()/1e9:.1f}GB", flush=True)

from collections import Counter

qs["cb_greedy"] = [truncate_answer(x) for x in greedy]
qs["cb_samples"] = [json.dumps([truncate_answer(x) for x in five], ensure_ascii=False)
                    for five in samp]

def major(five):
    c = Counter(truncate_answer(x) for x in five).most_common(1)[0]
    return c[0], c[1]                       # (다수 답, 득표수)
mj = [major(f) for f in samp]
qs["cb_major"], qs["major_votes"] = [m[0] for m in mj], [m[1] for m in mj]

qs["knows_new"]  = [score_row(p, g)[0] for p, g in zip(qs.cb_greedy, qs.gold_list)]
qs["knows_maj"]  = [score_row(p, g)[0] for p, g in zip(qs.cb_major,  qs.gold_list)]
qs["k_agree"]    = [sum(score_row(truncate_answer(x), g)[0] for x in five)/5
                    for five, g in zip(samp, qs.gold_list)]

qs["kside_new"] = np.where(qs.knows_new == 1, "K+", "K−")   # greedy  ← 주 라벨
qs["kside_maj"] = np.where(qs.knows_maj == 1, "K+", "K−")   # 다수결
qs["kside_shift"] = np.where(qs.major_votes < 3, "drop", qs.kside_maj)  # SHIFT 방식

qs["prompt_version"], qs["prompt_md5"] = PROMPT_VERSION, PROMPT_MD5
qs.to_parquet(P9 + "hidden_v2/k_relabel_v2.parquet", index=False)

print("greedy == major (문자열):", (qs.cb_greedy == qs.cb_major).mean().round(3))
print("kside_new == kside_maj  :", (qs.kside_new == qs.kside_maj).mean().round(3))
print(pd.crosstab(qs.kside_new, qs.kside_maj))
print("\nSHIFT drop (득표<3):", (qs.major_votes < 3).mean().round(3))
flip = (qs.kside != qs.kside_new)
print(f"구 라벨 대비 flip {flip.mean():.3f} ({flip.sum()}/{len(qs)})")
print(pd.crosstab(qs.kside, qs.kside_new))
print("불안정(0.2<k_agree<0.8):", round(((qs.k_agree>0.2)&(qs.k_agree<0.8)).mean(), 3))

612
  0/612 11s 16.1GB
  160/612 108s 16.1GB
  320/612 180s 16.1GB
  480/612 234s 16.1GB
greedy == major (문자열): 0.735
kside_new == kside_maj  : 0.951
kside_maj   K+   K−
kside_new          
K+         296   19
K−          11  286

SHIFT drop (득표<3): 0.25
구 라벨 대비 flip 0.155 (95/612)
kside_new   K+   K−
kside              
K+         299   79
K−          16  218
불안정(0.2<k_agree<0.8): 0.067


In [ ]:
## Cell K-2
m = df.merge(qs[["question","kside_new","kside_maj","k_agree","major_votes"]], on="question", how="left")
assert m.kside_new.notna().all()
m["cell_old"] = m.kside + m.ctx_label
m["cell_new"] = m.kside_new + m.ctx_label
m.to_csv(P9 + "hidden_v2/p8s_behavior_v2.csv", index=False, encoding="utf-8-sig")
print(pd.crosstab(m.cell_old, m.cell_new))

cell_new  K+C+  K+C-  K−C+  K−C-
cell_old                        
K+C+       299     0    79     0
K+C-         0   299     0    79
K−C+        16     0   218     0
K−C-         0    16     0   218


In [ ]:
## Cell K-3 — flip 원인 진단
fl = qs[(qs.kside=="K+") & (qs.kside_new=="K−")].copy()
fl["sub_hit"] = [score_row(p, g)[1] for p, g in zip(fl.cb_greedy, fl.gold_list)]
fl["words"]   = fl.cb_greedy.str.split().str.len()
print(f"K+→K− {len(fl)}건")
print(f"  gold 부분포함: {fl.sub_hit.mean():.3f}   ← 0.3 넘으면 채점 아티팩트")
print(f"  평균 단어수: {fl.words.mean():.2f}")
print(f"  k_agree 평균: {fl.k_agree.mean():.3f}")
print(fl[["question","cb_greedy","gold_list","k_agree"]].head(15).to_string())

print(f"\nK− 전체 평균 단어수: {qs[qs.kside_new=='K−'].cb_greedy.str.split().str.len().mean():.2f}")
print(f"K+ 전체 평균 단어수: {qs[qs.kside_new=='K+'].cb_greedy.str.split().str.len().mean():.2f}")

K+→K− 79건
  gold 부분포함: 0.241   ← 0.3 넘으면 채점 아티팩트
  평균 단어수: 3.14
  k_agree 평균: 0.114
                                                         question                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       cb_greedy                                                                                                                                                                                                           gold_list  k_agree
0    

In [ ]:
## Cell K-4 — 결함 정량화
import unicodedata as ud
from p8s_prompt import norm

def fold(s):
    return "".join(c for c in ud.normalize("NFD", str(s)) if ud.category(c) != "Mn")

fl["em_fold"] = [int(any(norm(fold(p)) == norm(fold(g)) for g in gl))
                 for p, gl in zip(fl.cb_greedy, fl.gold_list)]
fl["degen"]   = fl.words > 10
fl["city_q"]  = fl.question.str.contains("In what city", case=False)

print(f"발음부호만 고치면 회복: {fl.em_fold.sum()}건")
print(f"반복 붕괴(10단어 초과): {fl.degen.sum()}건")
print(f"'In what city' 질문:   {fl.city_q.sum()}건")

qs["em_fold"] = [int(any(norm(fold(p)) == norm(fold(g)) for g in gl))
                 for p, gl in zip(qs.cb_greedy, qs.gold_list)]
print(f"\n전체 K+ 변화: {qs.knows_new.sum()} → {qs.em_fold.sum()} "
      f"(+{qs.em_fold.sum() - qs.knows_new.sum()})")

pm = df.drop_duplicates("question").merge(qs[["question","kside_new"]], on="question")
pm["flip"] = pm.kside != pm.kside_new
print("\n" + pm.groupby("prop").flip.agg(["size","mean"]).round(3)
        .sort_values("mean", ascending=False).head(8).to_string())

발음부호만 고치면 회복: 1건
반복 붕괴(10단어 초과): 1건
'In what city' 질문:   5건

전체 K+ 변화: 315 → 316 (+1)

                size   mean
prop                       
color              5  0.400
genre             40  0.375
occupation        27  0.296
screenwriter      44  0.250
place of birth    39  0.205
father            46  0.196
producer          53  0.151
director          48  0.146


In [ ]:
## Cell K-5 — 방향별 조건부 비율 + 불안정 정의 교정
old_kp, old_km = (qs.kside=="K+"), (qs.kside=="K−")
print(f"P(K+→K− | old K+) = {(qs.kside_new[old_kp]=='K−').mean():.3f}  "
      f"({(qs.kside_new[old_kp]=='K−').sum()}/{old_kp.sum()})")
print(f"P(K−→K+ | old K−) = {(qs.kside_new[old_km]=='K+').mean():.3f}  "
      f"({(qs.kside_new[old_km]=='K+').sum()}/{old_km.sum()})")
print(f"순 K+ 감소 = {(qs.kside=='K+').sum()-(qs.kside_new=='K+').sum()}/612")

any_i  = ((qs.k_agree>0) & (qs.k_agree<1))
high_i = ((qs.k_agree>0.2) & (qs.k_agree<0.8))
print(f"\nany instability (0<k_agree<1): {any_i.mean():.3f} ({any_i.sum()}문항 / {any_i.sum()*2}행)")
print(f"high ambiguity (0.4~0.6)      : {high_i.mean():.3f} ({high_i.sum()}문항 / {high_i.sum()*2}행)")

# 다수결 정의 명확화 — 최빈 문자열 EM vs 샘플 EM 다수결
qs["kside_vote"] = np.where(qs.k_agree >= 0.6, "K+", "K−")
print(f"\nkside_new == kside_vote: {(qs.kside_new==qs.kside_vote).mean():.3f}")

# prop별 조건부 flip
pm = df.drop_duplicates("question").merge(qs[["question","kside_new"]], on="question")
t = pm[pm.kside=="K+"].groupby("prop").apply(
        lambda d: pd.Series({"n_oldK+": len(d), "P(→K−)": (d.kside_new=="K−").mean()}))
print("\n" + t.sort_values("P(→K−)", ascending=False).round(3).to_string())

P(K+→K− | old K+) = 0.209  (79/378)
P(K−→K+ | old K−) = 0.068  (16/234)
순 K+ 감소 = 63/612

any instability (0<k_agree<1): 0.152 (93문항 / 186행)
high ambiguity (0.4~0.6)      : 0.067 (41문항 / 82행)

kside_new == kside_vote: 0.941

                n_oldK+  P(→K−)
prop                           
producer           15.0   0.533
screenwriter       23.0   0.478
genre              30.0   0.467
occupation         17.0   0.412
color               5.0   0.400
place of birth     14.0   0.357
father             29.0   0.276
religion            8.0   0.250
author             46.0   0.196
composer           14.0   0.143
director           21.0   0.143
capital of         10.0   0.100
country            49.0   0.061
capital            65.0   0.046
sport              27.0   0.037
mother              5.0   0.000


/tmp/ipykernel_2282/2774803946.py:20: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  t = pm[pm.kside=="K+"].groupby("prop").apply(


In [ ]:
## Cell K-6 — flip 95건 semantic audit 시트
fa = qs[qs.kside != qs.kside_new][["question","cb_greedy","gold_list",
                                    "kside","kside_new","k_agree","cb_samples"]].copy()
fa["prop"] = fa.question.map(df.drop_duplicates("question").set_index("question")["prop"])
fa["label"] = ""     # WRONG / VALID_ALT / ALIAS / DEFECT / NOFORMAT / DEGEN
fa.to_csv(P9+"hidden_v2/k_flip_audit.csv", index=False, encoding="utf-8-sig")
print(len(fa))

95


In [ ]:
print(pd.crosstab(qs.kside_new, qs.kside_vote))
fa = pd.read_csv("k_flip_audit_labeled.csv")   # 내가 준 파일
m = qs.merge(fa[["question","label"]], on="question", how="left")
print(pd.crosstab(m.label, m.kside_vote))

kside_vote   K+   K−
kside_new           
K+          289   26
K−           10  287
kside_vote  K+  K−
label             
ALIAS        1   4
DEFECT       0   3
DEGEN        0   1
NOFORMAT     0   4
RECOVER     13   3
VALID_ALT    4  23
WRONG        2  37


In [ ]:
## Cell A — 완화 매칭 정의 + 영향 측정
import unicodedata as ud
from p8s_prompt import norm

def fold(s):
    return "".join(c for c in ud.normalize("NFD", str(s)) if ud.category(c) != "Mn")

def k_match(pred, golds):
    p = norm(fold(pred))
    if not p: return 0
    for x in [norm(fold(g)) for g in golds]:
        if x and (p == x or x in p or p in x):
            return 1
    return 0

qs["knows_k"] = [k_match(p, g) for p, g in zip(qs.cb_greedy, qs.gold_list)]
qs["kside_k"] = np.where(qs.knows_k == 1, "K+", "K−")

print("EM 라벨 vs 완화 라벨")
print(pd.crosstab(qs.kside_new, qs.kside_k))
print(f"\nK+ : {(qs.kside_new=='K+').sum()} → {(qs.kside_k=='K+').sum()}")

EM 라벨 vs 완화 라벨
kside_k     K+   K−
kside_new          
K+         315    0
K−          31  266

K+ : 315 → 346


In [ ]:
## Cell B — audit 라벨로 검증
fa = pd.read_csv("/content/k_flip_audit_labeled.csv")   # 먼저 코랩에 업로드
m = qs.merge(fa[["question","label"]], on="question", how="left")
print(pd.crosstab(m.label, m.kside_k))

# 새로 K+가 된 것 중 WRONG이 섞였는지 눈으로
new_kp = qs[(qs.kside_new=="K−") & (qs.kside_k=="K+")]
print(f"\n새로 K+ {len(new_kp)}건")
print(new_kp[["question","cb_greedy","gold_list"]].head(25).to_string())

kside_k    K+  K−
label            
ALIAS       2   3
DEFECT      0   3
DEGEN       0   1
NOFORMAT    4   0
RECOVER    16   0
VALID_ALT  17  10
WRONG       2  37

새로 K+ 31건
                                                                question                             cb_greedy                                                                                                                                                                                                                                                                                       gold_list
40                                                 What genre is Frifot?                                  Folk                                                                                                                                                                                     [traditional music, traditional folk music, popular melody, folk music, music traditions, musical folklore]
47                           

In [ ]:
## 저장
qs["kside_corr"] = qs.kside_k.copy()
fa = pd.read_csv("/content/k_flip_audit_labeled.csv")
fix = fa[fa.label.isin(["VALID_ALT","ALIAS","DEFECT"])].question
qs.loc[qs.question.isin(fix), "kside_corr"] = "K+"

qs.to_parquet(P9+"hidden_v2/k_relabel_v2.parquet", index=False)

m = df.merge(qs[["question","kside_k","kside_new","kside_corr","k_agree"]], on="question")
m["cell_k"] = m.kside_k + m.ctx_label
m.to_csv(P9+"hidden_v2/p8s_behavior_v2.csv", index=False, encoding="utf-8-sig")
print(m.cell_k.value_counts().to_dict())
print({c: (qs[c]=="K+").sum() for c in ["kside_new","kside_k","kside_corr"]})

{'K+C+': 346, 'K+C-': 346, 'K−C+': 266, 'K−C-': 266}
{'kside_new': np.int64(315), 'kside_k': np.int64(346), 'kside_corr': np.int64(362)}


In [ ]:
import hashlib
from google.colab import drive; drive.mount("/content/drive")
p = "/content/drive/MyDrive/Grad_share/data_share/P9/hidden_v2/p8s_v2_h_out.npy"
h = hashlib.md5()
with open(p, "rb") as f:
    for b in iter(lambda: f.read(1 << 24), b""): h.update(b)
print(h.hexdigest(), "← 기대 4a17f6de336b536209fa099dd7b29bf2")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
4a17f6de336b536209fa099dd7b29bf2 ← 기대 4a17f6de336b536209fa099dd7b29bf2
